# EF characterization workflow

This notebook switches to the `ge-ef-cr` configuration mode and walks
through the core `|e>` to `|f>` calibration steps: prepare the ge pulses,
obtain EF Rabi data, refine the EF drive frequency, and calibrate an EF
half-pi pulse.


## 1. Create an `Experiment` in `ge-ef-cr` mode


In [ ]:
import numpy as np

import qubex as qx

exp = qx.Experiment(
    system_id="YOUR_SYSTEM_ID",
    qubits=[0],
    configuration_mode="ge-ef-cr",
    # config_dir="/path/to/qubex-config/config",
    # params_dir="/path/to/qubex-config/params/YOUR_SYSTEM_ID",
)

Q0 = exp.qubit_labels[0]
print("target qubit:", Q0)

## 2. Connect and prepare the ge pulses


In [ ]:
exp.connect()

waveform_result = exp.check_waveform(targets=Q0, n_shots=512, plot=True)
rabi_result = exp.obtain_rabi_params(targets=Q0, n_shots=512, plot=True)
hpi_result = exp.calibrate_hpi_pulse(targets=Q0, n_shots=512, plot=True)

## 3. Measure the EF Rabi oscillation


In [ ]:
ef_rabi_result = exp.obtain_ef_rabi_params(
    targets=Q0,
    time_range=np.arange(0, 201, 4),
    n_shots=512,
    plot=True,
)

## 4. Calibrate the EF control frequency

The detuning window here is usually much narrower than the ge search.


In [ ]:
ef_frequency_result = exp.calibrate_ef_control_frequency(
    targets=Q0,
    detuning_range=np.linspace(-0.02, 0.02, 41),
    time_range=np.arange(0, 121, 4),
    n_shots=512,
    plot=True,
)

## 5. Re-run the EF Rabi scan after the frequency update


In [ ]:
ef_rabi_after_frequency = exp.obtain_ef_rabi_params(
    targets=Q0,
    time_range=np.arange(0, 201, 4),
    n_shots=512,
    plot=True,
)

## 6. Calibrate and inspect the EF half-pi pulse


In [ ]:
ef_hpi_result = exp.calibrate_ef_hpi_pulse(
    targets=Q0,
    n_rotations=1,
    n_shots=512,
    plot=True,
)

exp.ef_hpi_pulse[Q0].plot()

## 7. Optionally persist the updated note and reload


In [ ]:
# exp.calib_note.save()
# exp.reload()
